# gru_va campaign notebook — fused-L20, 102-bitstream sweep (REV1)
SENTINEL: NB102-REV1-2026-08-09

One notebook, two modes:
- **Section A** — interactive single-bitstream bring-up (same flow as the certified notebook, hardened)
- **Section B** — the full campaign loop over `campaign_manifest.csv` (resumable, per-row error capture)

Both share the config + helpers below. Run cells top-to-bottom through Section 2, then pick A or B.

## 1. Configuration — edit here, nowhere else

In [ ]:
import csv, hashlib, os, time
import numpy as np
from pynq import Overlay, allocate
from pynq.ps import Clocks

CAMPAIGN_DIR = "/home/xilinx/jupyter_notebooks/campaign102"
MANIFEST     = os.path.join(CAMPAIGN_DIR, "campaign_manifest.csv")
RESULTS      = os.path.join(CAMPAIGN_DIR, "campaign_results.csv")
GOLDEN_IN    = os.path.join(CAMPAIGN_DIR, "golden_input.bin")     # 4096 f32, shared stimulus
NAM_IN       = os.path.join(CAMPAIGN_DIR, "anchor_nam_in.f32")    # full-length real-music stimulus
IP_NAME      = "gru_va_0"
DMA_NAME     = "axi_dma_0"
CHUNK        = 65536         # locked for the campaign
REPS         = 200           # throughput window, matches certified cell 23
FCLK_MHZ     = 100.0
KEEP_FL_OUT  = True          # False = MD5 each 36 MB full-length output, then delete it
GOLDEN_N     = 4096
SR           = 48000.0

st = os.statvfs(CAMPAIGN_DIR)
print("free space: %.1f GB (full-length outputs need ~3.7 GB if KEEP_FL_OUT)" % (st.f_bavail*st.f_frsize/1e9))

## 2. Helpers (run once per kernel session)
Everything below bakes in the standing rules:
- `fclk0` re-applied + asserted after **every** `Overlay()` load (hwh reprograms 62.5 MHz)
- buffers freed **and reallocated** around every overlay load (freed PynqBuffers fail only on access)
- DMA transfer views sliced to `n` on partial chunks (the hang fix)
- no prints inside timed loops

In [ ]:
def md5_of(arr):
    return hashlib.md5(arr.tobytes()).hexdigest().upper()

def start_kernel(ip):
    ip.register_map.CTRL.AP_START = 1

def wait_done(ip, timeout_s=15.0):
    t0 = time.time()
    while int(ip.register_map.CTRL.AP_IDLE) != 1:
        if time.time() - t0 > timeout_s:
            raise TimeoutError("kernel not idle after %.1fs" % timeout_s)

def load_bitstream(bit_name):
    """Overlay load + clock fix + handle fetch, as one inseparable action."""
    ol = Overlay(os.path.join(CAMPAIGN_DIR, bit_name))
    Clocks.fclk0_mhz = FCLK_MHZ
    assert abs(Clocks.fclk0_mhz - FCLK_MHZ) < 0.5, "fclk0 set failed: %s" % Clocks.fclk0_mhz
    ip  = getattr(ol, IP_NAME)
    dma = getattr(ol, DMA_NAME)
    return ol, ip, dma

def run_golden(ip, dma, golden_x, ref=None):
    """4096-sample gate. Returns (board_out, row_fragment dict)."""
    frag = {}
    ibuf = allocate(shape=(GOLDEN_N,), dtype=np.float32)
    obuf = allocate(shape=(GOLDEN_N,), dtype=np.float32)
    try:
        ibuf[:] = golden_x; ibuf.flush()
        rm = ip.register_map
        rm.mode = 1; rm.n_samples = GOLDEN_N; rm.reset_state = 1
        dma.recvchannel.transfer(obuf); start_kernel(ip); dma.sendchannel.transfer(ibuf)
        dma.sendchannel.wait(); dma.recvchannel.wait(); wait_done(ip)
        obuf.invalidate()
        bg = np.asarray(obuf).copy()
    finally:
        ibuf.freebuffer(); obuf.freebuffer()
    frag["golden_md5"] = md5_of(bg)
    if ref is not None:
        d = np.abs(bg - ref)
        frag["max_abs_err"] = "%.6e" % float(d.max())
        frag["worst_idx"]   = str(int(d.argmax()))
        frag["gate"] = "PASS" if np.array_equal(bg, ref) else "FAIL"
    else:
        frag["gate"] = "NO_REF"
    return bg, frag

def run_throughput(ip, dma, ibc, obc, nam_x):
    """Warm chunk + REPS timed reps. Returns row_fragment dict. No prints inside."""
    rm = ip.register_map
    ibc[:] = nam_x[:CHUNK]; ibc.flush()
    rm.mode = 1; rm.n_samples = CHUNK; rm.reset_state = 1
    dma.recvchannel.transfer(obc); start_kernel(ip); dma.sendchannel.transfer(ibc)
    dma.sendchannel.wait(); dma.recvchannel.wait(); wait_done(ip)
    rm.reset_state = 0
    hw = 0.0
    t0 = time.time()
    for _ in range(REPS):
        dma.recvchannel.transfer(obc)
        start_kernel(ip)
        dma.sendchannel.transfer(ibc)
        ta = time.perf_counter()
        dma.sendchannel.wait()
        hw += time.perf_counter() - ta
        dma.recvchannel.wait()
        wait_done(ip)
    dt = time.time() - t0
    us_tot = 1e6*dt/(REPS*CHUNK); us_hw = 1e6*hw/(REPS*CHUNK)
    return {"us_total": "%.4f" % us_tot, "us_hw": "%.4f" % us_hw,
            "cyc_total": "%.2f" % (us_tot*FCLK_MHZ), "cyc_hw": "%.2f" % (us_hw*FCLK_MHZ),
            "ksamp_s": "%.0f" % (1e3/us_tot), "x_rt": "%.2f" % (1e6/us_tot/SR)}

def run_full_length(ip, dma, ibc, obc, nam_x, out_fl):
    """State-carried chunked run over the whole stimulus. Returns row_fragment dict."""
    rm = ip.register_map
    n_total = nam_x.size
    t0 = time.time(); pos = 0; ci = 0
    while pos < n_total:
        n = min(CHUNK, n_total - pos)
        ibc[:n] = nam_x[pos:pos+n]; ibc.flush()
        rm.mode = 1; rm.n_samples = n; rm.reset_state = 1 if ci == 0 else 0
        dma.recvchannel.transfer(obc[:n] if n < CHUNK else obc)
        start_kernel(ip)
        dma.sendchannel.transfer(ibc[:n] if n < CHUNK else ibc)
        dma.sendchannel.wait(); dma.recvchannel.wait(); wait_done(ip)
        obc.invalidate()
        out_fl[pos:pos+n] = obc[:n]
        pos += n; ci += 1
    fl_dt = time.time() - t0
    return {"fl_secs": "%.2f" % fl_dt,
            "fl_ksamp_s": "%.0f" % (n_total/fl_dt/1e3),
            "fl_x_rt": "%.1f" % (n_total/fl_dt/SR),
            "fl_md5": md5_of(out_fl)}

FIELDS = ["timestamp","cell","width","bit","status","gate","max_abs_err","worst_idx",
          "golden_md5","cyc_total","cyc_hw","us_total","us_hw","ksamp_s","x_rt",
          "fl_secs","fl_ksamp_s","fl_x_rt","fl_md5","fl_out_file","fclk_mhz","note"]

def append_row(row):
    new = not os.path.exists(RESULTS)
    with open(RESULTS, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        if new: w.writeheader()
        w.writerow(row); f.flush(); os.fsync(f.fileno())

def load_done_set():
    done = set()
    if os.path.exists(RESULTS):
        with open(RESULTS) as f:
            for r in csv.DictReader(f):
                if r.get("status") == "OK":
                    done.add((r["cell"], int(r["width"])))
    return done

golden_x = np.fromfile(GOLDEN_IN, dtype=np.float32)
assert golden_x.size == GOLDEN_N, "golden_input.bin wrong size: %d" % golden_x.size
nam_x = np.fromfile(NAM_IN, dtype=np.float32)
out_fl = np.empty(nam_x.size, dtype=np.float32)
print("helpers defined; golden %d samples, nam %d samples" % (golden_x.size, nam_x.size))

---
## Section A — interactive single-bitstream mode
Use this for debugging one (cell, width) or reproducing yesterday's bring-up flow. Skip to Section B for the campaign.

In [ ]:
# A1. Pick one bitstream and load it
BIT = "gru_L20_rodent_max_w20.bit"       # EDIT: any manifest bit name
ol, ip, dma = load_bitstream(BIT)
print("loaded:", BIT, "| fclk0 =", Clocks.fclk0_mhz, "MHz")
print("IP keys:", list(ol.ip_dict.keys()))
print(ip.register_map)

In [ ]:
# A2. Golden gate — digits vs float golden; byte-compare only if a csim ref is staged
REF = ""    # digits-only mode. To byte-compare: set to "ref_<cell>_w<W>.f32" matching A1
ref = None
if REF:
    rp = os.path.join(CAMPAIGN_DIR, REF)
    if os.path.exists(rp) and os.path.getsize(rp) == 4*GOLDEN_N:
        ref = np.fromfile(rp, dtype=np.float32)
    else:
        print("!! ref file missing or wrong size -- falling back to digits-only")
bg, frag = run_golden(ip, dma, golden_x, ref)
print(frag)
print("first 5 outputs:", bg[:5])

# cross-check vs FLOAT golden (always available on this board)
fg = np.fromfile("/home/xilinx/jupyter_notebooks/golden_output.bin", dtype=np.float32)
d = np.abs(bg - fg)
print("vs FLOAT golden: max %.6e at idx %d" % (d.max(), d.argmax()))
print("  (rodent_max w20 certified digits: 2.316236e-04 at 2093;")
print("   other cells/widths differ -- compare against their grid row)")

# gate=PASS (byte-compare path) means bit-identical to that (cell,width) csim
# output -- 0.0 is correct ONLY against that reference, never the float golden.

In [ ]:
# A3. Throughput window (identical structure to certified cell 23)
ibc = allocate(shape=(CHUNK,), dtype=np.float32)
obc = allocate(shape=(CHUNK,), dtype=np.float32)
print(run_throughput(ip, dma, ibc, obc, nam_x))

In [ ]:
# A4. Full-length run + save (reuses A3 buffers)
frag = run_full_length(ip, dma, ibc, obc, nam_x, out_fl)
print(frag)
name = "board_nam_" + BIT.replace("gru_L20_", "").replace(".bit", "") + ".f32"
out_fl.tofile(os.path.join(CAMPAIGN_DIR, name))
print("wrote %s -- verify byte size (%d) + date before scoring" % (name, 4*nam_x.size))

In [ ]:
# A5. Free Section A buffers before loading another bitstream
try:
    ibc.freebuffer(); obc.freebuffer()
except Exception:
    pass
print("Section A buffers freed -- reallocate (rerun A3) after any new load_bitstream")

---
## Section B — campaign loop over the manifest
- Resumable: rows already `status=OK` in `campaign_results.csv` are skipped. Restart the cell after any interruption.
- Per-row try/except: a bad bitstream logs `ERROR`/`TIMEOUT` and the loop continues (next overlay load resets the fabric).
- Known limitation: a true DMA-channel hang blocks inside `.wait()` — Kernel ▸ Interrupt, rerun the cell, resume skips completed rows.
- Budget: ~45–60 s/row ≈ 1.5–2 h for 102. One heartbeat line per row; nothing prints inside timed loops.

In [ ]:
# B1. Load manifest + show resume status (safe to run any time)
with open(MANIFEST) as f:
    entries = [{"cell": r["cell"], "width": int(r["width"]),
                "bit": r["bit"], "ref": r.get("ref", "")}
               for r in csv.DictReader(f)]
done = load_done_set()
todo = [e for e in entries if (e["cell"], e["width"]) not in done]
print("manifest: %d rows | done: %d | to run: %d" % (len(entries), len(done), len(todo)))
print("estimated wall time: %.1f h at 55 s/row" % (len(todo)*55/3600.0))

In [ ]:
# B2. THE CAMPAIGN LOOP -- start it and leave it alone
for i, e in enumerate(todo):
    row = {k: "" for k in FIELDS}
    row.update(timestamp=time.strftime("%Y-%m-%d %H:%M:%S"),
               cell=e["cell"], width=e["width"], bit=e["bit"])
    tag = "%s_w%d" % (e["cell"], e["width"])
    ibc = obc = None
    try:
        ol, ip, dma = load_bitstream(e["bit"])
        row["fclk_mhz"] = "%.1f" % Clocks.fclk0_mhz

        ref = None
        if e["ref"]:
            rp = os.path.join(CAMPAIGN_DIR, e["ref"])
            if os.path.exists(rp) and os.path.getsize(rp) == 4*GOLDEN_N:
                ref = np.fromfile(rp, dtype=np.float32)
        bg, frag = run_golden(ip, dma, golden_x, ref)
        row.update(frag)
        bg.tofile(os.path.join(CAMPAIGN_DIR, "board_gold_%s.f32" % tag))

        ibc = allocate(shape=(CHUNK,), dtype=np.float32)
        obc = allocate(shape=(CHUNK,), dtype=np.float32)
        row.update(run_throughput(ip, dma, ibc, obc, nam_x))
        row.update(run_full_length(ip, dma, ibc, obc, nam_x, out_fl))

        fl_name = "board_nam_%s.f32" % tag
        fl_path = os.path.join(CAMPAIGN_DIR, fl_name)
        out_fl.tofile(fl_path)
        sz = os.path.getsize(fl_path)
        if sz != 4*nam_x.size:
            row["note"] = "FL_SIZE_%d_EXPECTED_%d" % (sz, 4*nam_x.size)
        if KEEP_FL_OUT:
            row["fl_out_file"] = fl_name
        else:
            os.remove(fl_path); row["fl_out_file"] = "DELETED_MD5_ONLY"
        row["status"] = "OK"
    except TimeoutError as ex:
        row.update(status="TIMEOUT", note=str(ex))
    except Exception as ex:
        row.update(status="ERROR", note=type(ex).__name__ + ": " + str(ex)[:120])
    finally:
        for b in (ibc, obc):
            try:
                if b is not None: b.freebuffer()
            except Exception:
                pass
    append_row(row)
    print("[%s] %d/%d %-22s %-7s gate=%-6s %s k/s  fl=%s s" % (
        time.strftime("%H:%M"), i+1, len(todo), tag, row["status"],
        row.get("gate","-"), row.get("ksamp_s","-"), row.get("fl_secs","-")))
print("campaign pass complete -- results in", RESULTS)

In [ ]:
# B3. Results summary -- gate verdicts and throughput spread
rows = list(csv.DictReader(open(RESULTS)))
ok = [r for r in rows if r["status"] == "OK"]
print("rows: %d | OK: %d | not OK: %d" % (len(rows), len(ok), len(rows)-len(ok)))
for r in rows:
    if r["status"] != "OK" or r.get("gate") == "FAIL":
        print("  !!", r["cell"], "w"+r["width"], r["status"], r.get("gate",""), r.get("note",""))
if ok:
    ch = [float(r["cyc_hw"]) for r in ok if r["cyc_hw"]]
    print("cyc_hw  min/max: %.2f / %.2f  (pre-registered: flat ~142.5)" % (min(ch), max(ch)))
    ks = [float(r["ksamp_s"]) for r in ok if r["ksamp_s"]]
    print("ksamp/s min/max: %.0f / %.0f" % (min(ks), max(ks)))

## Cleanup
Nothing persistent to free — Section B frees its buffers per row. If Section A buffers are live, run A5.